# Speculative Decoding with Interruptible Streaming

In [2]:
import openai
import asyncio

In [3]:
client = openai.AsyncOpenAI(base_url="http://127.0.0.1:5000/v1", api_key="0608da5d28eb10cea2914f3de0f3ddba")

In [ ]:
async def generate_response(prompt, stop_event):
    """Streams response but cancels immediately if stop_event is set."""
    try:
        response = await client.chat.completions.create(
            model="cognitivecomputations_dolphin-2.9-llama3-8b",
            messages=[{"role": "user", "content": prompt}],
            stream=True
        )
        async for chunk in response:
            if stop_event.is_set():  # Stop streaming if interrupted
                print("\n[Interrupted]\n")
                stop_event.clear()  # Reset event for next run
                await response.close()
                return
            if chunk.choices[0].delta.content is None: break

            print(chunk.choices[0].delta.content, end="")
    except asyncio.CancelledError:
        print("\n[Generation Cancelled]\n")

async def prewritten_stream():
    stop_event = asyncio.Event()
    current_task = None
    
    # Simulated input changes
    prompts = [
        "Tell me about the history of space exploration.",
        "Actually, focus on the Apollo missions.",
        "Wait, just tell me about the Apollo 11 moon landing."
    ]
    
    for new_prompt in prompts:
        if current_task and not current_task.done():
            stop_event.set()
            await asyncio.sleep(0.01)
        
        print(f"\n[New Input Detected: \"{new_prompt}\"]\n[Generating Response...]\n")
        current_task = asyncio.create_task(generate_response(new_prompt, stop_event))
        await asyncio.sleep(2)

await prewritten_stream()



[New Input Detected: "Tell me about the history of space exploration."]
[Generating Response...]

Space exploration is the exploration of space beyond Earth's atmosphere. It has been conducted by various countries and organizations, with the goal of acquiring scientific knowledge, developing new technologies, and extending human presence into space. The history of space exploration can be divided into several key stages.

1. Early ideas and attempts: The idea of traveling into space can be traced back to early science fiction and fantasy literature, with authors like Jules Verne and H.G. Wells writing about journeys to the moon and other celestial bodies.
[New Input Detected: "Actually, focus on the Apollo missions."]
[Generating Response...]


[Interrupted]

Sure, let's talk about the Apollo missions. The Apollo program was a series of crewed space missions undertaken by NASA, the United States' space agency.

The goal of the Apollo program was to land humans on the Moon and bring th

 in science, technology, and international collaborations.